# Single-threshold directional change

This first notebook builds the completed-event ledger for one directional-change threshold, plots the events on price, and describes their size, duration, velocity, and overshoot. It fits no model; the filtered event-time HMM starts in `1_threshold_dc.ipynb`.

For each completed extreme-to-extreme trend:

$$\mathrm{TMV} = \frac{|P_{\mathrm{EXT},n} - P_{\mathrm{EXT},n-1}|}{P_{\mathrm{EXT},n-1}\,\theta}$$

$$T = t_{\mathrm{EXT},n} - t_{\mathrm{EXT},n-1}$$

$$R = \frac{\mathrm{TMV}\,\theta}{T}$$

The ending extreme is identified retrospectively. Therefore, TMV, T, and R are not available at `ext_end_idx`; they first become knowable at the later DC confirmation, `dcc_idx`. The notebook records that timestamp as `information_time`.

In [1]:
import numpy as np
import pandas as pd

from dc_basics import prepare_event_ledger
from dc_basics.core import parse_dc_events
from dc_basics.data import resolve_data_root
from dc_basics.plotting import plot_dc_indicator_distributions, plot_directional_change

## Configuration

In [2]:
DATA_ROOT = resolve_data_root()  # $DC_DATA_ROOT or the git-ignored .dc_data_root file
SYMBOL = "US500.r"
DATA_PATH = DATA_ROOT / f"{SYMBOL}.parquet"

THETA = 0.003
N_PRICES = 5_000

print(f"Symbol: {SYMBOL}")
print(f"Data: {DATA_PATH}")
print(f"Threshold: {THETA:.4%}")
print(f"Chart window: {N_PRICES:,} prices")

Symbol: US500.r
Data: D:\data_dump\market_data\raw\FXPIG-Server\5\US500.r.parquet
Threshold: 0.3000%
Chart window: 5,000 prices


## Load and audit the bars

Directional-change parsing is path-dependent, so the price series must be chronological, unique, finite, and strictly positive. The cell below refuses unsorted, duplicated, non-finite, or non-positive prices rather than silently repairing them. If `time` is stored as a column rather than the Parquet index, it is promoted to the index explicitly.

In [3]:
raw = pd.read_parquet(DATA_PATH)
if not isinstance(raw.index, pd.DatetimeIndex):
    if "time" not in raw.columns:
        raise TypeError("Expected a DatetimeIndex or a 'time' column.")
    raw = raw.set_index(pd.to_datetime(raw.pop("time")))

raw = raw.sort_index()
if raw.index.has_duplicates:
    raise ValueError(f"Found {raw.index.duplicated().sum():,} duplicate timestamps.")
if "close" not in raw.columns:
    raise KeyError("The input data has no 'close' column.")

prices = raw["close"].astype(np.float64).rename(SYMBOL)
if not np.isfinite(prices.to_numpy()).all():
    raise ValueError("Close contains NaN or infinite values.")
if prices.le(0.0).any():
    raise ValueError("DC thresholds require strictly positive prices.")

print(f"Bars: {len(prices):,}")
print(f"Start: {prices.index[0]}")
print(f"End: {prices.index[-1]}")
print(f"Timezone: {prices.index.tz}")
raw.tail()

Bars: 245,467


Start: 2023-05-22 14:00:00+02:00
End: 2026-11-09 13:30:00+02:00
Timezone: pytz.FixedOffset(120)


,open,high,low,close,tick_volume,spread
date,,,,,,
2026-11-09 13:10:00+02:00,7425.52,7427.97,7420.02,7420.52,1099.0,41
2026-11-09 13:15:00+02:00,7420.52,7440.15,7418.32,7432.52,1494.0,41
2026-11-09 13:20:00+02:00,7432.52,7436.14,7427.27,7429.02,1296.0,41
2026-11-09 13:25:00+02:00,7429.27,7433.14,7427.52,7429.90,857.0,41
2026-11-09 13:30:00+02:00,7429.90,7433.77,7429.52,7432.27,677.0,41


## Calculate completed directional-change events

`parse_dc_events` returns one row per completed trend. `prepare_event_ledger` then attaches the event identity and `information_time = dcc_idx`, because that confirmation bar is when the ending extreme and its completed TMV, T, and R become knowable. It also checks the ledger's causal invariants: no two events share a confirmation bar, no ending extreme lies after its information time, and $R = \mathrm{TMV}\,\theta/T$ on every row.

The ledger states direction explicitly. `direction` (identical to `completed_leg_direction`) is the leg that **ended**. `dcc_direction` is the direction of the confirmation itself, and `new_live_mode` is the mode that is live from `information_time` on. A completed `up` leg is confirmed by a downward DCC and switches the live mode down.

The library's string ID is kept as `ledger_event_id` (for example `E17`). This lab also keeps the plain event number as `event_id`, because its plots place events on a numeric event-order axis.

In [4]:
source_events = parse_dc_events(prices=prices, theta=THETA)
if source_events.empty:
    raise ValueError("No completed DC events were found. Check the data and threshold.")

events = prepare_event_ledger(source_events, prefix="E", scale="single", theta=THETA)
events = events.rename(columns={"event_id": "ledger_event_id"})
events.insert(0, "event_id", np.arange(1, len(events) + 1, dtype=np.int64))

event_columns = [
    "event_id", "threshold", "direction", "dcc_direction", "new_live_mode",
    "ext_start_idx", "ext_start_price", "ext_end_idx", "ext_end_price", "dcc_idx", "dcc_price",
    "information_time", "tmv", "t", "r",
]

print(f"Completed events: {len(events):,}")
events[event_columns].tail(10)

Completed events: 3,862


,event_id,threshold,direction,ext_start_idx,ext_start_price,ext_end_idx,ext_end_price,dcc_idx,dcc_price,information_time,tmv,t,r
3852,3853,0.003,up,2026-11-06 09:30:00+02:00,7335.90,2026-11-06 11:35:00+02:00,7401.27,2026-11-06 12:05:00+02:00,7378.02,2026-11-06 12:05:00+02:00,2.970324,25,0.000356
3853,3854,0.003,down,2026-11-06 11:35:00+02:00,7401.27,2026-11-06 15:25:00+02:00,7357.72,2026-11-06 16:35:00+02:00,7395.22,2026-11-06 16:35:00+02:00,1.961375,46,0.000128
3854,3855,0.003,up,2026-11-06 15:25:00+02:00,7357.72,2026-11-06 17:10:00+02:00,7433.97,2026-11-06 17:30:00+02:00,7397.72,2026-11-06 17:30:00+02:00,3.454422,21,0.000493
3855,3856,0.003,down,2026-11-06 17:10:00+02:00,7433.97,2026-11-06 17:30:00+02:00,7397.72,2026-11-06 17:45:00+02:00,7426.82,2026-11-06 17:45:00+02:00,1.625421,4,0.001219
3856,3857,0.003,up,2026-11-06 17:30:00+02:00,7397.72,2026-11-06 18:50:00+02:00,7431.07,2026-11-06 19:35:00+02:00,7392.82,2026-11-06 19:35:00+02:00,1.502715,16,0.000282
3857,3858,0.003,down,2026-11-06 18:50:00+02:00,7431.07,2026-11-06 19:50:00+02:00,7384.32,2026-11-06 20:30:00+02:00,7413.62,2026-11-06 20:30:00+02:00,2.097051,12,0.000524
3858,3859,0.003,up,2026-11-06 19:50:00+02:00,7384.32,2026-11-06 22:05:00+02:00,7414.12,2026-11-06 22:50:00+02:00,7378.12,2026-11-06 22:50:00+02:00,1.345193,27,0.000149
3859,3860,0.003,down,2026-11-06 22:05:00+02:00,7414.12,2026-11-09 01:55:00+02:00,7374.65,2026-11-09 02:55:00+02:00,7402.14,2026-11-09 02:55:00+02:00,1.774542,34,0.000157
3860,3861,0.003,up,2026-11-09 01:55:00+02:00,7374.65,2026-11-09 06:45:00+02:00,7432.15,2026-11-09 07:50:00+02:00,7409.40,2026-11-09 07:50:00+02:00,2.598993,58,0.000134
3861,3862,0.003,down,2026-11-09 06:45:00+02:00,7432.15,2026-11-09 09:50:00+02:00,7373.90,2026-11-09 10:20:00+02:00,7402.27,2026-11-09 10:20:00+02:00,2.612524,37,0.000212


## Visualize the last N prices

The close is shown with completed extreme-to-extreme trends and their later DC confirmations. A turning-point marker describes where an extreme occurred; its hover text shows when that extreme became knowable. The final unfinished event is intentionally absent because `parse_dc_events` emits completed events only.

In [5]:
fig = plot_directional_change(prices=prices, events=events, n_prices=N_PRICES, title=f"{SYMBOL} | theta={THETA:.4%}",)
fig.show()

## Descriptive EDA of TMV, T, and R

The mean answers the average question, but these event indicators are typically right-skewed. Median and tail quantiles are reported beside it so a small number of extreme events cannot silently determine the interpretation. These are descriptive event-level results: consecutive DC events are path-dependent, so the rows must not be treated as independent observations for significance testing.

In [6]:
INDICATORS = ["tmv", "t", "r"]
QUANTILES = [0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]

overall_summary = events[INDICATORS].describe(percentiles=QUANTILES).T
overall_summary.index.name = "indicator"
overall_summary.style.format(precision=6)

,count,mean,std,min,10%,25%,50%,75%,90%,95%,99%,max
indicator,,,,,,,,,,,,
tmv,3862.000000,2.398213,1.552848,1.000390,1.130821,1.373348,1.912538,2.897117,4.209480,5.253013,8.067023,21.481858
t,3862.000000,63.547385,90.439018,1.000000,5.000000,11.000000,28.000000,75.000000,179.900000,240.000000,415.340000,1100.000000
r,3862.000000,0.000509,0.000920,0.000009,0.000044,0.000085,0.000214,0.000502,0.001215,0.002026,0.004531,0.011954


### Direction-conditioned averages

The event count must be read with every directional comparison. A difference between up and down averages is descriptive here; it is not yet evidence of a stable or tradable asymmetry.

In [7]:
direction_summary = events.groupby("direction", sort=False).agg(
    event_count=("event_id", "size"),
    tmv_mean=("tmv", "mean"),
    tmv_median=("tmv", "median"),
    t_mean=("t", "mean"),
    t_median=("t", "median"),
    r_mean=("r", "mean"),
    r_median=("r", "median"),
)
direction_summary.style.format({
    "event_count": "{:,}",
    "tmv_mean": "{:.6f}",
    "tmv_median": "{:.6f}",
    "t_mean": "{:.2f}",
    "t_median": "{:.2f}",
    "r_mean": "{:.6e}",
    "r_median": "{:.6e}",
})

,event_count,tmv_mean,tmv_median,t_mean,t_median,r_mean,r_median
direction,,,,,,,
up,"1,931",2.459193,1.931890,73.58,29.00,5.020026e-04,2.003685e-04
down,"1,931",2.337233,1.893983,53.51,26.00,5.156014e-04,2.323405e-04


### Direction-conditioned distributions

Each histogram is normalized within direction, so up and down shapes remain comparable even if their event counts differ. The initial x-axis view ends at the full-sample 99th percentile; Plotly's **Autoscale** button reveals all tail observations.

In [8]:
distribution_fig = plot_dc_indicator_distributions(events)
distribution_fig.show()

## Overshoot: the part of a leg beyond its directional-change confirmation

A completed leg has two parts.

- The **directional-change portion** runs from the starting extreme to the DCC that opened the leg's mode. It is at least $\theta$ by construction, and more when a sampled close leaps past the exact boundary.
- The **overshoot** runs from that opening DCC to the ending extreme.

The opening DCC of row $n$ is row $n-1$'s `dcc_*` columns, so the first row has no observed overshoot. In threshold units:

$$\mathrm{DC}_n = \frac{|P_{\mathrm{DCC},n-1} - P_{\mathrm{EXT},n-1}|}{P_{\mathrm{EXT},n-1}\,\theta}, \qquad \mathrm{OS}_n = \frac{|P_{\mathrm{EXT},n} - P_{\mathrm{DCC},n-1}|}{P_{\mathrm{DCC},n-1}\,\theta}$$

The two parts compound: $(1 \pm \mathrm{DC}_n\theta)(1 \pm \mathrm{OS}_n\theta) = 1 \pm \mathrm{TMV}_n\theta$, with $+$ for up legs and $-$ for down legs, so $\mathrm{TMV} \approx \mathrm{DC} + \mathrm{OS}$. The cell checks that identity on every row.

Like TMV, T, and R, the **completed** overshoot is known only at the leg's own confirming DCC (`information_time`). While the leg is live, only the overshoot so far is observable, and it may still grow.

In [ ]:
opening_dcc_price = events["dcc_price"].shift(1)
opening_dcc_pos = events["dcc_pos"].shift(1)
leg_sign = pd.Series(np.where(events["direction"].eq("up"), 1.0, -1.0), index=events.index)

events["opening_dcc_idx"] = events["dcc_idx"].shift(1)
events["dc_portion_theta"] = leg_sign * (opening_dcc_price / events["ext_start_price"] - 1.0) / THETA
events["overshoot_theta"] = leg_sign * (events["ext_end_price"] / opening_dcc_price - 1.0) / THETA
events["overshoot_bars"] = events["ext_end_pos"] - opening_dcc_pos

has_opening_dcc = opening_dcc_price.notna()
linked_events = events.loc[has_opening_dcc]
linked_sign = leg_sign.loc[has_opening_dcc]

# Consecutive legs share an extreme, and each leg's opening DCC lies inside the leg.
assert (events["ext_start_pos"].iloc[1:].to_numpy() == events["ext_end_pos"].iloc[:-1].to_numpy()).all()
assert (opening_dcc_pos.loc[has_opening_dcc] > linked_events["ext_start_pos"]).all()
assert (opening_dcc_pos.loc[has_opening_dcc] <= linked_events["ext_end_pos"]).all()
assert (linked_events["dc_portion_theta"] >= 1.0 - 1e-9).all()
assert (linked_events["overshoot_theta"] >= -1e-9).all()
np.testing.assert_allclose(
    (1.0 + linked_sign * linked_events["dc_portion_theta"] * THETA)
    * (1.0 + linked_sign * linked_events["overshoot_theta"] * THETA),
    1.0 + linked_sign * linked_events["tmv"] * THETA,
    rtol=1e-10,
)

overshoot_summary = linked_events.groupby("direction", sort=False).agg(
    event_count=("event_id", "size"),
    tmv_median=("tmv", "median"),
    dc_portion_median=("dc_portion_theta", "median"),
    overshoot_mean=("overshoot_theta", "mean"),
    overshoot_median=("overshoot_theta", "median"),
    overshoot_p90=("overshoot_theta", lambda values: values.quantile(0.90)),
    zero_overshoot_share=("overshoot_theta", lambda values: float((values <= 1e-12).mean())),
    overshoot_bars_median=("overshoot_bars", "median"),
)
print(f"Events without an observed opening DCC (the first event): {int((~has_opening_dcc).sum())}")
display(overshoot_summary.style.format({
    "event_count": "{:,}",
    "tmv_median": "{:.3f}",
    "dc_portion_median": "{:.3f}",
    "overshoot_mean": "{:.3f}",
    "overshoot_median": "{:.3f}",
    "overshoot_p90": "{:.3f}",
    "zero_overshoot_share": "{:.2%}",
    "overshoot_bars_median": "{:.1f}",
}))
events.loc[has_opening_dcc, [
    "event_id", "direction", "opening_dcc_idx", "ext_end_idx", "information_time",
    "tmv", "dc_portion_theta", "overshoot_theta", "overshoot_bars",
]].tail(10)

## How many bars does a completed leg take? Counts of T = 1, 2, 3, …

$T$ counts bars between the two extremes. A large share of very small $T$ means the threshold is close to the bar-scale noise of this market: legs complete almost as soon as they start. The table counts each $T$ up to `MAX_LISTED_T` and groups the rest, by completed-leg direction and in total.

In [ ]:
MAX_LISTED_T = 10

t_bucket = events["t"].astype(int).astype(str).where(events["t"] <= MAX_LISTED_T, f">{MAX_LISTED_T}")
bucket_order = [str(value) for value in range(1, MAX_LISTED_T + 1)] + [f">{MAX_LISTED_T}"]
t_counts = pd.crosstab(t_bucket, events["direction"]).reindex(bucket_order, fill_value=0)
t_counts["all"] = t_counts.sum(axis=1)
t_counts["share"] = t_counts["all"] / len(events)
t_counts["cumulative_share"] = t_counts["share"].cumsum()
t_counts.index.name = "T (bars)"
assert t_counts["all"].sum() == len(events)
t_counts.style.format({"share": "{:.2%}", "cumulative_share": "{:.2%}"})

## Find events whose extremes are consecutive observations

`T = 1` means the starting and ending extremes occupy adjacent rows in the stored close series. `elapsed_minutes` checks whether those observations are also one 5-minute period apart in wall-clock time. `confirmation_delay_bars` measures how many subsequent observations were needed to confirm the ending extreme.

In [9]:
consecutive_events = events.loc[events["t"] == 1].copy()
consecutive_events["elapsed_minutes"] = (consecutive_events["ext_end_idx"] - consecutive_events["ext_start_idx"]).dt.total_seconds() / 60.0
consecutive_events["observed_move_pct"] = (consecutive_events["ext_end_price"] / consecutive_events["ext_start_price"] - 1.0) * 100.0
consecutive_events["confirmation_delay_bars"] = (consecutive_events["dcc_pos"] - consecutive_events["ext_end_pos"])

consecutive_columns = [
    "event_id", "direction", "ext_start_idx", "ext_start_price",
    "ext_end_idx", "ext_end_price", "elapsed_minutes", "observed_move_pct",
    "tmv", "r", "dcc_idx", "confirmation_delay_bars",
]

print(f"Events with T=1: {len(consecutive_events):,} of {len(events):,}")
consecutive_events[consecutive_columns]

Events with T=1: 63 of 3,862


,event_id,direction,ext_start_idx,ext_start_price,ext_end_idx,ext_end_price,elapsed_minutes,observed_move_pct,tmv,r,dcc_idx,confirmation_delay_bars
617,618,down,2024-02-02 19:35:00+02:00,4960.44,2024-02-02 19:40:00+02:00,4941.44,5.0,-0.383031,1.276768,0.003830,2024-02-02 20:15:00+02:00,7
691,692,down,2024-03-20 21:25:00+02:00,5202.49,2024-03-20 21:30:00+02:00,5186.14,5.0,-0.314273,1.047575,0.003143,2024-03-20 21:40:00+02:00,2
923,924,down,2024-07-19 00:45:00+02:00,5562.82,2024-07-19 00:50:00+02:00,5544.77,5.0,-0.324476,1.081586,0.003245,2024-07-19 03:05:00+02:00,15
992,993,up,2024-08-02 18:30:00+02:00,5358.32,2024-08-02 18:35:00+02:00,5381.32,5.0,0.429239,1.430797,0.004292,2024-08-02 18:55:00+02:00,4
1017,1018,down,2024-08-05 18:35:00+02:00,5174.47,2024-08-05 18:40:00+02:00,5158.92,5.0,-0.300514,1.001713,0.003005,2024-08-05 18:45:00+02:00,1
...,...,...,...,...,...,...,...,...,...,...,...,...
3452,3453,up,2026-06-11 19:15:00+02:00,7293.39,2026-06-11 19:20:00+02:00,7315.37,5.0,0.301369,1.004563,0.003014,2026-06-11 19:55:00+02:00,7
3818,3819,up,2026-10-29 22:30:00+02:00,7689.72,2026-10-29 22:35:00+02:00,7715.72,5.0,0.338114,1.127046,0.003381,2026-10-29 22:40:00+02:00,1
3819,3820,down,2026-10-29 22:35:00+02:00,7715.72,2026-10-29 22:40:00+02:00,7688.97,5.0,-0.346695,1.155649,0.003467,2026-10-29 22:45:00+02:00,1
3838,3839,up,2026-11-05 17:50:00+02:00,7424.32,2026-11-05 17:55:00+02:00,7448.57,5.0,0.326629,1.088764,0.003266,2026-11-05 18:00:00+02:00,1


### Inspect one consecutive-extreme event on the raw bars

Change `EVENT_ID` to inspect another row from the table above. The displayed path begins at the starting extreme and ends at the later confirmation bar.

In [10]:
if consecutive_events.empty:
    raise ValueError("No T=1 events are available at the current threshold.")

EVENT_ID = int(consecutive_events.iloc[-1]["event_id"])
selected_event = events.loc[events["event_id"] == EVENT_ID].iloc[0]

event_bars = raw.iloc[
    int(selected_event["ext_start_pos"]):int(selected_event["dcc_pos"]) + 1
].copy()
event_bars["dc_landmark"] = ""
event_bars.loc[selected_event["ext_start_idx"], "dc_landmark"] = "start EXT"
event_bars.loc[selected_event["ext_end_idx"], "dc_landmark"] = "end EXT"
event_bars.loc[selected_event["dcc_idx"], "dc_landmark"] = "DCC confirmation"

print(
    f"Event {EVENT_ID}: {selected_event['direction']} | "
    f"T={selected_event['t']} | TMV={selected_event['tmv']:.4f} | R={selected_event['r']:.3e}"
)
event_bars[["open", "high", "low", "close", "dc_landmark"]]

Event 3840: down | T=1 | TMV=1.0517 | R=3.155e-03


,open,high,low,close,dc_landmark
date,,,,,
2026-11-05 17:55:00+02:00,7424.44,7455.82,7417.57,7448.57,start EXT
2026-11-05 18:00:00+02:00,7448.07,7451.07,7423.62,7425.07,end EXT
2026-11-05 18:05:00+02:00,7425.07,7449.77,7418.70,7447.32,
2026-11-05 18:10:00+02:00,7446.32,7476.32,7445.82,7473.07,DCC confirmation
